# Segmentação Semântica em Imagens Histológicas

Esse notebook explora 36 configurações para o dataset de tecido tumoral (OCDC, Mendeley), configurado em:

- 3 Combinações de Arquitetura/Modo: Sharp U-Net (FS), U-Net ResNet-18 (FS), U-Net ResNet-18 (PT-ALL)
- 2 Funções de Perda: BCE (`BCEWithLogitsLoss`) vs Dice Loss
- 2 Condições de Augmentation: Sem Augmentation vs Com Augmentation (geométrica sincronizada)
- 3 Seeds: 42, 123, 2025

Totalizando 36 experimentos.

In [1]:
!pip install -q segmentation-models-pytorch albumentations thop

import os
import gc
import json
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from thop import profile, clever_format

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo em uso: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.0 MB/s eta 0:00:00
Dispositivo em uso: cuda
GPU: Tesla T4
VRAM Total: 15.64 GB


#### Montagem do Drive e Definição dos Caminhos do Repositório

In [2]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/projeto 3/Repositorio'
DATASET_DIR = os.path.join(BASE_DIR, 'datasets/tumor_tissue')
TRAIN_DIR = os.path.join(DATASET_DIR, 'train/tumor/patch/640x640')
TEST_DIR = os.path.join(DATASET_DIR, 'test/tumor/patch/640x640')
SPLITS_CSV = os.path.join(DATASET_DIR, 'splits_tumoral.csv')

RESULTS_DIR = os.path.join(BASE_DIR, 'results')
CURVES_DIR = os.path.join(RESULTS_DIR, 'curves_json')
PLOTS_DIR = os.path.join(RESULTS_DIR, 'plots_pdf')
QUALITATIVE_DIR = os.path.join(RESULTS_DIR, 'qualitative')
CHECKPOINTS_DIR = os.path.join(BASE_DIR, 'checkpoints')
CONSOLIDATED_CSV = os.path.join(RESULTS_DIR, 'resultados_tumoral.csv')

for d in [RESULTS_DIR, CURVES_DIR, PLOTS_DIR, QUALITATIVE_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)
print('Diretórios verificados!')

Mounted at /content/drive
Diretórios verificados!


#### Configurações de Parametros do Experimento

In [3]:
SPLIT_SEED = 42
SEEDS = [42, 123, 2025]

INPUT_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-4
PATIENCE = 12  # early stopping

ARCHITECTURES = [
    {'model': 'sharp_unet', 'encoder': 'none', 'training_mode': 'FS'},
    {'model': 'unet_resnet18', 'encoder': 'resnet18', 'training_mode': 'FS'},
    {'model': 'unet_resnet18', 'encoder': 'resnet18', 'training_mode': 'PT-ALL'}
]
LOSSES = ['bce', 'dice']
AUGMENTATIONS = ['sem_aug', 'com_aug']

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SPLIT_SEED)

#### Splits Oficiais e Divisão Treino/Validação por Lâmina

In [4]:
def generate_tumoral_splits(train_dir, test_dir, output_csv, seed=42):
    if os.path.exists(output_csv):
        print(f'Arquivo de splits já existe: {output_csv}')
        return pd.read_csv(output_csv)

    def list_pairs(root):
        records = []
        rel_root = os.path.relpath(root, DATASET_DIR)
        for slide_id in sorted(os.listdir(root)):
            img_dir = os.path.join(root, slide_id, '01-roi', '01-original')
            mask_dir = os.path.join(root, slide_id, '01-roi', '02-mask')
            if not (os.path.isdir(img_dir) and os.path.isdir(mask_dir)):
                continue
            for img_name in sorted(os.listdir(img_dir)):
                if not img_name.lower().endswith('.png'):
                    continue
                if os.path.exists(os.path.join(mask_dir, img_name)):
                    records.append({
                        'image': os.path.join(rel_root, slide_id, '01-roi', '01-original', img_name),
                        'mask': os.path.join(rel_root, slide_id, '01-roi', '02-mask', img_name),
                        'slide_id': slide_id
                    })
        return records

    df_test = pd.DataFrame(list_pairs(test_dir))
    df_test['split'] = 'test'

    df_train_pool = pd.DataFrame(list_pairs(train_dir))
    unique_slides = df_train_pool['slide_id'].unique()

    rng = np.random.default_rng(seed)
    rng.shuffle(unique_slides)

    n_train = int(len(unique_slides) * 0.80)
    train_slides = set(unique_slides[:n_train])

    df_train_pool['split'] = df_train_pool['slide_id'].apply(lambda s: 'train' if s in train_slides else 'val')

    df = pd.concat([df_train_pool, df_test], ignore_index=True)
    df.to_csv(output_csv, index=False)
    print(f'Splits gerados. Distribuição: {df["split"].value_counts().to_dict()}')
    return df

#### Dataset e Transformações de Augmentation Sincronizadas

In [5]:
class TumoralTissueDataset(Dataset):
    def __init__(self, df, dataset_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.dataset_dir = dataset_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.dataset_dir, row['image'])
        mask_path = os.path.join(self.dataset_dir, row['mask'])
        image = np.array(Image.open(img_path).convert('RGB'))
        mask = np.array(Image.open(mask_path).convert('L'), dtype=np.float32)

        # tumor vem como pixel claro (>= 128) contra fundo escuro
        mask = (mask >= 128).astype(np.float32)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        if not isinstance(mask, torch.Tensor):
            mask = torch.tensor(mask, dtype=torch.float32)
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        return image, mask

# normalização fixa do ImageNet
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)

def get_transforms(use_augmentation=False, img_size=INPUT_SIZE):
    if use_augmentation:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Normalize(mean=NORM_MEAN, std=NORM_STD),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=NORM_MEAN, std=NORM_STD),
            ToTensorV2()
        ])

#### Arquiteturas Sharp U-Net e U-Net com Encoder ResNet-18

In [6]:
# bloco de sharpening aplicado nas skip connections
class SharpBlock(nn.Module):
    """
    Bloco de sharpening do Sharp U-Net (Zunair & Ben Hamza, 2021)
    """
    def __init__(self, in_channels):
        super(SharpBlock, self).__init__()
        # Eq. 1: kernel Laplaciano de 8 vizinhos
        kernel = torch.tensor([[-1., -1., -1.],
                                [-1.,  8., -1.],
                                [-1., -1., -1.]], dtype=torch.float32)
        kernel = kernel.repeat(in_channels, 1, 1, 1)
        # kernel fixo: nao eh atualizado durante o treinamento
        self.weight = nn.Parameter(kernel, requires_grad=False)
        self.in_channels = in_channels

    def forward(self, x):
        # Eq. 2: S = I + K * I (convolucao seguida de soma explicita)
        conv_out = F.conv2d(x, self.weight, padding=1, groups=self.in_channels)
        return x + conv_out


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


class SharpUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.sharp_blocks = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        curr_in = in_channels
        for feat in features:
            self.downs.append(ConvBlock(curr_in, feat))
            self.sharp_blocks.append(SharpBlock(feat))
            curr_in = feat

        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)

        for feat in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feat * 2, feat, kernel_size=2, stride=2))
            self.ups.append(ConvBlock(feat * 2, feat))

        self.final_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for i, down in enumerate(self.downs):
            x = down(x)
            sharp_skip = self.sharp_blocks[i](x)
            skip_connections.append(sharp_skip)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skip_connections[idx // 2]
            if x.shape != skip.shape:
                x = TF.resize(x, size=skip.shape[2:])
            concat_x = torch.cat((skip, x), dim=1)
            x = self.ups[idx + 1](concat_x)

        return self.final_conv(x)


# escolhe qual modelo montar
def build_model(model_name, training_mode, in_channels=3, num_classes=1):
    if model_name == 'sharp_unet':
        return SharpUNet(in_channels=in_channels, num_classes=num_classes)
    elif model_name == 'unet_resnet18':
        weights = 'imagenet' if training_mode == 'PT-ALL' else None
        return smp.Unet(
            encoder_name='resnet18',
            encoder_weights=weights,
            in_channels=in_channels,
            classes=num_classes
        )
    else:
        raise ValueError(f'Modelo desconhecido: {model_name}')

#### Funções de Perda e Métricas de Segmentação

In [7]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice


def get_loss_function(loss_name):
    if loss_name.lower() == 'bce':
        return nn.BCEWithLogitsLoss()
    elif loss_name.lower() == 'dice':
        return DiceLoss()
    else:
        raise ValueError(f'Loss desconhecida: {loss_name}')


@torch.no_grad()
def calculate_metrics(preds_binary, targets, smooth=1e-6):
    # preds_binary e targets: [B, 1, H, W] binários (0 ou 1)
    p_fg = preds_binary.view(-1)
    t_fg = targets.view(-1)

    p_bg = 1.0 - p_fg
    t_bg = 1.0 - t_fg

    tp_fg = (p_fg * t_fg).sum().item()
    fp_fg = (p_fg * (1.0 - t_fg)).sum().item()
    fn_fg = ((1.0 - p_fg) * t_fg).sum().item()

    dice_fg = (2.0 * tp_fg + smooth) / (2.0 * tp_fg + fp_fg + fn_fg + smooth)
    iou_fg = (tp_fg + smooth) / (tp_fg + fp_fg + fn_fg + smooth)
    prec_fg = (tp_fg + smooth) / (tp_fg + fp_fg + smooth)
    rec_fg = (tp_fg + smooth) / (tp_fg + fn_fg + smooth)

    tp_bg = (p_bg * t_bg).sum().item()
    fp_bg = (p_bg * (1.0 - t_bg)).sum().item()
    fn_bg = ((1.0 - p_bg) * t_bg).sum().item()

    dice_bg = (2.0 * tp_bg + smooth) / (2.0 * tp_bg + fp_bg + fn_bg + smooth)
    iou_bg = (tp_bg + smooth) / (tp_bg + fp_bg + fn_bg + smooth)

    mDice = (dice_fg + dice_bg) / 2.0
    mIoU = (iou_fg + iou_bg) / 2.0

    return {
        'dice_background': dice_bg,
        'dice_foreground': dice_fg,
        'mDice': mDice,
        'iou_background': iou_bg,
        'iou_foreground': iou_fg,
        'mIoU': mIoU,
        'precision_foreground': prec_fg,
        'recall_foreground': rec_fg
    }

#### Funções de Treino, Validação e Avaliação

In [8]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_masks = [], []

    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)
        running_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(outputs)
        preds_binary = (probs >= 0.5).float()

        all_preds.append(preds_binary.cpu())
        all_masks.append(masks.cpu())

    all_preds = torch.cat(all_preds, dim=0)
    all_masks = torch.cat(all_masks, dim=0)

    metrics = calculate_metrics(all_preds, all_masks)
    metrics['loss'] = running_loss / len(loader.dataset)
    return metrics


def profile_model_complexity(model, input_size=(1, 3, 256, 256), device='cpu'):
    model_eval = model.to(device).eval()
    dummy_input = torch.randn(input_size).to(device)
    try:
        macs, params = profile(model_eval, inputs=(dummy_input,), verbose=False)
        gflops = (macs * 2) / 1e9
    except Exception:
        params = sum(p.numel() for p in model.parameters())
        gflops = 0.0

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params, round(gflops, 3)

#### Execução de Experimento Individual

In [9]:
def run_single_experiment(cfg, df_splits, dataset_dir, consolidated_csv, curves_dir, checkpoints_dir, device):
    run_id = f"{cfg['dataset']}_{cfg['model']}_{cfg['training_mode']}_{cfg['loss']}_{cfg['augmentation']}_seed{cfg['seed']}"

    if os.path.exists(consolidated_csv):
        res_df = pd.read_csv(consolidated_csv)
        if 'run_id' in res_df.columns and run_id in res_df['run_id'].values:
            print(f'Run já feita: {run_id} -- pulando')
            return

    print(f'\n--- {run_id} ---')

    set_seed(cfg['seed'])

    use_aug = (cfg['augmentation'] == 'com_aug')
    train_tf = get_transforms(use_augmentation=use_aug, img_size=cfg['input_size'])
    eval_tf = get_transforms(use_augmentation=False, img_size=cfg['input_size'])

    train_df = df_splits[df_splits['split'] == 'train']
    val_df = df_splits[df_splits['split'] == 'val']
    test_df = df_splits[df_splits['split'] == 'test']

    train_ds = TumoralTissueDataset(train_df, dataset_dir, transform=train_tf)
    val_ds = TumoralTissueDataset(val_df, dataset_dir, transform=eval_tf)
    test_ds = TumoralTissueDataset(test_df, dataset_dir, transform=eval_tf)

    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=2)

    model = build_model(cfg['model'], cfg['training_mode']).to(device)
    criterion = get_loss_function(cfg['loss'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    num_params, trainable_params, gflops = profile_model_complexity(model, (1, 3, cfg['input_size'], cfg['input_size']), device)

    best_val_mDice = -1.0
    best_epoch = 0
    patience_counter = 0
    best_model_path = os.path.join(checkpoints_dir, f'best_{run_id}.pth')

    history = {'train_loss': [], 'val_loss': [], 'val_mDice': [], 'val_mIoU': []}

    for epoch in range(1, cfg['epochs'] + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = eval_epoch(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_mDice'].append(val_metrics['mDice'])
        history['val_mIoU'].append(val_metrics['mIoU'])

        scheduler.step(val_metrics['mDice'])

        # salva o melhor checkpoint pelo mDice de validação
        if val_metrics['mDice'] > best_val_mDice:
            best_val_mDice = val_metrics['mDice']
            best_epoch = epoch
            torch.save(model.state_dict(), best_model_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'Early stopping na época {epoch}. Melhor época: {best_epoch}')
                break

        if epoch % 5 == 0 or epoch == 1:
            print(f'  [Época {epoch:02d}/{cfg["epochs"]}] Train Loss: {train_loss:.4f} | Val Loss: {val_metrics["loss"]:.4f} | Val mDice: {val_metrics["mDice"]:.4f}')

    json_path = os.path.join(curves_dir, f'{run_id}.json')
    with open(json_path, 'w') as f:
        json.dump(history, f, indent=2)

    # recarrega o melhor checkpoint e avalia no teste
    model.load_state_dict(torch.load(best_model_path))
    test_metrics = eval_epoch(model, test_loader, criterion, device)

    result_row = {
        'run_id': run_id,
        'repetition': cfg['seed_idx'],
        'seed': cfg['seed'],
        'dataset': cfg['dataset'],
        'task': cfg['task'],
        'model': cfg['model'],
        'encoder': cfg['encoder'],
        'training_mode': cfg['training_mode'],
        'augmentation': cfg['augmentation'],
        'loss': cfg['loss'],
        'input_size': cfg['input_size'],
        'epochs': cfg['epochs'],
        'batch_size': cfg['batch_size'],
        'dice_background_test': round(test_metrics['dice_background'], 4),
        'dice_foreground_test': round(test_metrics['dice_foreground'], 4),
        'mDice_test': round(test_metrics['mDice'], 4),
        'iou_background_test': round(test_metrics['iou_background'], 4),
        'iou_foreground_test': round(test_metrics['iou_foreground'], 4),
        'mIoU_test': round(test_metrics['mIoU'], 4),
        'precision_foreground_test': round(test_metrics['precision_foreground'], 4),
        'recall_foreground_test': round(test_metrics['recall_foreground'], 4),
        'num_params': num_params,
        'trainable_params': trainable_params,
        'gflops': gflops,
        'best_epoch': best_epoch,
        'val_mDice_best': round(best_val_mDice, 4)
    }

    row_df = pd.DataFrame([result_row])
    if os.path.exists(consolidated_csv):
        row_df.to_csv(consolidated_csv, mode='a', header=False, index=False)
    else:
        row_df.to_csv(consolidated_csv, mode='w', header=True, index=False)

    print(f'Test mDice: {test_metrics["mDice"]:.4f} | Test mIoU: {test_metrics["mIoU"]:.4f}')

    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

#### Execução

Monta as 36 combinações e roda uma por uma e quem já está no CSV consolidado é pulado.

In [10]:
df_splits = generate_tumoral_splits(TRAIN_DIR, TEST_DIR, SPLITS_CSV, seed=SPLIT_SEED)

configs_tumoral = []
for arch in ARCHITECTURES:
    for loss in LOSSES:
        for aug in AUGMENTATIONS:
            for s_idx, seed in enumerate(SEEDS, 1):
                configs_tumoral.append({
                    'dataset': 'tumoral',
                    'task': 'tumor_vs_background',
                    'model': arch['model'],
                    'encoder': arch['encoder'],
                    'training_mode': arch['training_mode'],
                    'loss': loss,
                    'augmentation': aug,
                    'seed': seed,
                    'seed_idx': s_idx,
                    'input_size': INPUT_SIZE,
                    'epochs': EPOCHS,
                    'batch_size': BATCH_SIZE,
                    'lr': LR
                })

print(f'Total de configurações geradas para Tecido Tumoral: {len(configs_tumoral)}')

for cfg in configs_tumoral:
    run_single_experiment(cfg, df_splits, DATASET_DIR, CONSOLIDATED_CSV, CURVES_DIR, CHECKPOINTS_DIR, device)

print('Fim das 36 execuções de Tecido Tumoral.')

Arquivo de splits já existe: /content/drive/MyDrive/projeto 3/Repositorio/datasets/tumor_tissue/splits_tumoral.csv
Total de configurações geradas para Tecido Tumoral: 36
Run já feita: tumoral_sharp_unet_FS_bce_sem_aug_seed42 -- pulando
Run já feita: tumoral_sharp_unet_FS_bce_sem_aug_seed123 -- pulando
Run já feita: tumoral_sharp_unet_FS_bce_sem_aug_seed2025 -- pulando
Run já feita: tumoral_sharp_unet_FS_bce_com_aug_seed42 -- pulando
Run já feita: tumoral_sharp_unet_FS_bce_com_aug_seed123 -- pulando
Run já feita: tumoral_sharp_unet_FS_bce_com_aug_seed2025 -- pulando
Run já feita: tumoral_sharp_unet_FS_dice_sem_aug_seed42 -- pulando
Run já feita: tumoral_sharp_unet_FS_dice_sem_aug_seed123 -- pulando
Run já feita: tumoral_sharp_unet_FS_dice_sem_aug_seed2025 -- pulando
Run já feita: tumoral_sharp_unet_FS_dice_com_aug_seed42 -- pulando
Run já feita: tumoral_sharp_unet_FS_dice_com_aug_seed123 -- pulando
Run já feita: tumoral_sharp_unet_FS_dice_com_aug_seed2025 -- pulando
Run já feita: tumora

#### Tabela Resumo dos Resultados

In [11]:
if os.path.exists(CONSOLIDATED_CSV):
    df_res = pd.read_csv(CONSOLIDATED_CSV)
    print('Resumo das métricas no conjunto de teste (média e desvio padrão):')
    summary = df_res.groupby(['model', 'training_mode', 'loss', 'augmentation']).agg({
        'mDice_test': ['mean', 'std'],
        'mIoU_test': ['mean', 'std'],
        'dice_foreground_test': ['mean', 'std'],
        'precision_foreground_test': ['mean', 'std'],
        'recall_foreground_test': ['mean', 'std'],
        'gflops': 'first',
        'num_params': 'first'
    }).reset_index()
    display(summary)
else:
    print('Treinamentos não executados.')

Resumo das métricas no conjunto de teste (média e desvio padrão):


model training_mode  loss augmentation mDice_test            \
                                                         mean       std   
0      sharp_unet            FS   bce      com_aug   0.831333  0.049623   
1      sharp_unet            FS   bce      sem_aug   0.888367  0.023330   
2      sharp_unet            FS  dice      com_aug   0.862133  0.019951   
3      sharp_unet            FS  dice      sem_aug   0.860100  0.038832   
4   unet_resnet18            FS   bce      com_aug   0.837200  0.053293   
5   unet_resnet18            FS   bce      sem_aug   0.828567  0.018601   
6   unet_resnet18            FS  dice      com_aug   0.829933  0.051313   
7   unet_resnet18            FS  dice      sem_aug   0.784200  0.045111   
8   unet_resnet18        PT-ALL   bce      com_aug   0.907000  0.010736   
9   unet_resnet18        PT-ALL   bce      sem_aug   0.886867  0.001877   
10  unet_resnet18        PT-ALL  dice      com_aug   0.902967  0.007158   
11  unet_resnet18        PT-ALL  dice      sem_aug   0.883267  0.013603   

   mIoU_test           dice_foreground_test            \
        mean       std                 mean       std   
0   0.713567  0.072256             0.821967  0.040557   
1   0.800067  0.038052             0.872600  0.028069   
2   0.758067  0.031208             0.854100  0.015682   
3   0.756000  0.060557             0.853200  0.032666   
4   0.722433  0.078687             0.829767  0.050304   
5   0.708067  0.027306             0.809133  0.014614   
6   0.711633  0.073657             0.823833  0.042360   
7   0.646633  0.062139             0.786333  0.037340   
8   0.830167  0.017895             0.894733  0.011880   
9   0.797000  0.003081             0.873967  0.001739   
10  0.823367  0.011972             0.891933  0.006313   
11  0.791333  0.021802             0.870367  0.012564   

   precision_foreground_test           recall_foreground_test            \
                        mean       std                   mean       std   
0                   0.783567  0.096767               0.871900  0.034047   
1                   0.887767  0.023863               0.859000  0.047350   
2                   0.802700  0.048909               0.915300  0.029216   
3                   0.804567  0.080791               0.913933  0.034779   
4                   0.773300  0.071750               0.896700  0.021546   
5                   0.802867  0.046657               0.817233  0.018506   
6                   0.766367  0.077603               0.894600  0.009822   
7                   0.698367  0.054749               0.901700  0.024759   
8                   0.902267  0.021039               0.887700  0.018644   
9                   0.864233  0.021652               0.884767  0.025769   
10                  0.882200  0.025892               0.902533  0.016581   
11                  0.858533  0.033588               0.883300  0.011177   

    gflops num_params  
     first      first  
0   27.492    7767361  
1   27.492    7767361  
2   27.492    7767361  
3   27.492    7767361  
4   10.855   14328209  
5   10.855   14328209  
6   10.855   14328209  
7   10.855   14328209  
8   10.855   14328209  
9   10.855   14328209  
10  10.855   14328209  
11  10.855   14328209